# Loading the Libraries for the Prediction Model 

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, log_loss

# Web Scraping

In [ ]:
URL = "https://www.transfermarkt.com/weltmeisterschaft/teilnehmer/pokalwettbewerb/FIWC/saison_id/2025"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=HEADERS)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

teams = []

for table in soup.select("table.items"):
    header_cells = [th.get_text(strip=True) for th in table.select("thead th")]
    header_cells = [h for h in header_cells if h]

    for tr in table.select("tbody tr"):
        cells = tr.find_all("td")
        if not cells:
            continue

        club_link = tr.find("a", href=lambda h: h and "/startseite/verein/" in h)
        if not club_link:
            continue

        texts = [td.get_text(strip=True) for td in cells if td.get_text(strip=True)]

        row = {
            "Club": club_link.get_text(strip=True),
        }

        n = min(len(header_cells), len(texts))
        for h, v in zip(header_cells[-n:], texts[-n:]):
            row[h] = v

        teams.append(row)

df = pd.DataFrame(teams)
print(df)
df.to_csv("world_cup_2026_participants.csv", index=False)

# Loading the Data and Cleaning

In [2]:
Elo = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\elo_ratings_wc2026.csv")
WC_2026_Squads = pd.read_csv("Data\\world_cup_2026_participants.csv",  encoding='latin-1')
Goals = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\goalscorers.csv")
Results = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\results.csv")
Shootouts = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\shootouts.csv")
Schedule = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\world-cup-2026-schedule.csv")

In [3]:
team_dictionary = {
    'Czech Republic': 'Czechia',                              
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',            
    'Democratic Republic of the Congo': 'DR Congo',            
    'Turkiye': 'Turkey',                                       
    'Cabo Verde': 'Cape Verde',                                
    'Congo DR': 'DR Congo',                                   
    'Côte d’Ivoire': 'Ivory Coast',                            
    'Korea Republic': 'South Korea',                           
    'Türkiye': 'Turkey',    
    'United States': 'USA',   
    'Congo DR' : 'DR Congo',
    "Curaçao": "Curacao",                             
}

In [ ]:
# Grabbing the first 48 rows of the squads data to ensure we have only the participating teams.
WC_2026_Squads_Cleaned = WC_2026_Squads.iloc[0:48]

WC_2026_Squads_Cleaned['Club'] = WC_2026_Squads_Cleaned['Club'].str.strip().replace(team_dictionary)

WC_2026_Squads_Cleaned = WC_2026_Squads_Cleaned.rename(columns={
    'Club': 'team',
    '&oslash-Age': 'Average Age(In Euros)',
    '&oslash-Market Value': 'Average Market Value(In Euros)',
})

WC_2026_Squads_Cleaned['Foreigners'] = (
    WC_2026_Squads_Cleaned['Foreigners'].str.rstrip(' %').astype(float) / 100
)

def parse_value(x):
    x = x.replace('â¬', '').strip()
    if x.endswith('bn'):
        return float(x[:-2]) * 1_000_000_000
    if x.endswith('m'):
        return float(x[:-1]) * 1_000_000
    if x.endswith('k'):
        return float(x[:-1]) * 1_000
    return float(x)

WC_2026_Squads_Cleaned['Market Value'] = WC_2026_Squads_Cleaned['Market Value'].apply(parse_value)
WC_2026_Squads_Cleaned['Average Market Value(In Euros)'] = WC_2026_Squads_Cleaned['Average Market Value(In Euros)'].apply(parse_value)

print(WC_2026_Squads_Cleaned.head())

In [ ]:
Results['date'] = pd.to_datetime(Results['date'])

Results['home_team'] = Results['home_team'].str.strip().replace(team_dictionary)
Results['away_team'] = Results['away_team'].str.strip().replace(team_dictionary)

Results = Results[Results['home_score'].notna()].copy()
Results['home_score'] = Results['home_score'].astype(int)
Results['away_score'] = Results['away_score'].astype(int)

print(Results.head())

In [ ]:
Schedule['date'] = pd.to_datetime(Schedule['date'])

Schedule['team_a'] = Schedule['team_a'].str.strip().replace(team_dictionary)
Schedule['team_b'] = Schedule['team_b'].str.strip().replace(team_dictionary)

print(Schedule.head())

In [ ]:
Goals['date'] = pd.to_datetime(Goals['date'])

Goals['home_team'] = Goals['home_team'].str.strip().replace(team_dictionary)
Goals['away_team'] = Goals['away_team'].str.strip().replace(team_dictionary)
Goals['team'] = Goals['team'].str.strip().replace(team_dictionary)

print(Goals.head())

In [ ]:
Shootouts['date'] = pd.to_datetime(Shootouts['date'])

Shootouts['home_team'] = Shootouts['home_team'].str.strip().replace(team_dictionary)
Shootouts['away_team'] = Shootouts['away_team'].str.strip().replace(team_dictionary)
Shootouts['winner'] = Shootouts['winner'].str.strip().replace(team_dictionary)

print(Shootouts.head())

In [9]:
Elo['snapshot_date'] = pd.to_datetime(Elo['snapshot_date'])
Elo['country'] = Elo['country'].str.strip().replace(team_dictionary)

Elo_Pre_WC = Elo[Elo['snapshot_date'] == '2026-05-27'].copy

# Joining Elo and Results

In [ ]:
WC2026_Nations = set(Elo['country'].unique())

Results_filter = Results[
    Results['home_team'].isin(WC2026_Nations) &
    Results['away_team'].isin(WC2026_Nations)
].copy()

Elo_Filter = (
    Elo[['country', 'snapshot_date', 'rating']]
    .sort_values('snapshot_date')
    .reset_index(drop=True)
)

Results_sorted = Results_filter.sort_values('date').reset_index(drop=True)

Results_with_elo = pd.merge_asof(
    Results_sorted,
    Elo_Filter.rename(columns={'country': 'home_team', 'rating': 'home_elo', 'snapshot_date': 'home_snap'}),
    left_on='date',
    right_on='home_snap',
    by='home_team',
    direction='backward'
).drop(columns='home_snap')

Results_with_elo = pd.merge_asof(
    Results_with_elo,
    Elo_Filter.rename(columns={'country': 'away_team', 'rating': 'away_elo', 'snapshot_date': 'away_snap'}),
    left_on='date',
    right_on='away_snap',
    by='away_team',
    direction='backward'
).drop(columns='away_snap')

Results_with_elo['elo_diff'] = Results_with_elo['home_elo'] - Results_with_elo['away_elo']

Results_with_elo = Results_with_elo.dropna(subset=['home_elo', 'away_elo'])
Results_with_elo['date'] = pd.to_datetime(Results_with_elo['date'])

Results_Elo = Results_with_elo[Results_with_elo['date'].dt.year >= 1990]

# Feature Engineering